In [1]:
!nvidia-smi

Mon Apr 27 05:28:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile assignment.cu

#include <iostream>
#include <chrono>
#include <cstdlib>

using namespace std;
using namespace std::chrono;

// ---------------- VECTOR ADDITION ----------------
__global__ void vectorAddCUDA(int *a, int *b, int *c, int n)
{
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if (idx < n)
        c[idx] = a[idx] + b[idx];
}

void vectorAddCPU(int *a, int *b, int *c, int n)
{
    for (int i = 0; i < n; i++)
        c[i] = a[i] + b[i];
}

// ---------------- MATRIX MULTIPLICATION ----------------
__global__ void matrixMulCUDA(int *a, int *b, int *c, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int sum = 0;
        for (int k = 0; k < N; k++)
            sum += a[row * N + k] * b[k * N + col];
        c[row * N + col] = sum;
    }
}

void matrixMulCPU(int *a, int *b, int *c, int N)
{
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
        {
            int sum = 0;
            for (int k = 0; k < N; k++)
                sum += a[i * N + k] * b[k * N + j];
            c[i * N + j] = sum;
        }
}

// ---------------- MAIN ----------------
int main()
{
    const int vecSize = 1 << 24;
    const int matrixSize = 1024;

    // ---------------- VECTOR ADDITION ----------------
    int *h_a = new int[vecSize];
    int *h_b = new int[vecSize];
    int *h_c_cpu = new int[vecSize];
    int *h_c_gpu = new int[vecSize];

    for (int i = 0; i < vecSize; i++)
    {
        h_a[i] = rand() % 100;
        h_b[i] = rand() % 100;
    }

    // CPU timing
    auto start = high_resolution_clock::now();
    vectorAddCPU(h_a, h_b, h_c_cpu, vecSize);
    auto end = high_resolution_clock::now();
    cout << "[Vector Addition - CPU] Time: "
         << duration_cast<milliseconds>(end - start).count() << " ms\n";

    // GPU memory
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, vecSize * sizeof(int));
    cudaMalloc(&d_b, vecSize * sizeof(int));
    cudaMalloc(&d_c, vecSize * sizeof(int));

    // GPU timing (INCLUDING memory transfer)
    start = high_resolution_clock::now();

    cudaMemcpy(d_a, h_a, vecSize * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, vecSize * sizeof(int), cudaMemcpyHostToDevice);

    vectorAddCUDA<<<(vecSize + 255) / 256, 256>>>(d_a, d_b, d_c, vecSize);
    cudaDeviceSynchronize();

    cudaMemcpy(h_c_gpu, d_c, vecSize * sizeof(int), cudaMemcpyDeviceToHost);

    end = high_resolution_clock::now();

    cout << "[Vector Addition - GPU] Time: "
         << duration_cast<milliseconds>(end - start).count() << " ms\n\n";

    // ---------------- MATRIX MULTIPLICATION ----------------
    int *matA = new int[matrixSize * matrixSize];
    int *matB = new int[matrixSize * matrixSize];
    int *matC_cpu = new int[matrixSize * matrixSize];
    int *matC_gpu = new int[matrixSize * matrixSize];

    for (int i = 0; i < matrixSize * matrixSize; i++)
    {
        matA[i] = rand() % 100;
        matB[i] = rand() % 100;
    }

    // CPU timing
    start = high_resolution_clock::now();
    matrixMulCPU(matA, matB, matC_cpu, matrixSize);
    end = high_resolution_clock::now();
    cout << "[Matrix Multiplication - CPU] Time: "
         << duration_cast<milliseconds>(end - start).count() << " ms\n";

    // GPU memory
    int *d_matA, *d_matB, *d_matC;
    size_t bytes = matrixSize * matrixSize * sizeof(int);

    cudaMalloc(&d_matA, bytes);
    cudaMalloc(&d_matB, bytes);
    cudaMalloc(&d_matC, bytes);

    dim3 threadsPerBlock(16, 16);
    dim3 blocksPerGrid((matrixSize + 15) / 16, (matrixSize + 15) / 16);

    // GPU timing (INCLUDING memory transfer)
    start = high_resolution_clock::now();

    cudaMemcpy(d_matA, matA, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_matB, matB, bytes, cudaMemcpyHostToDevice);

    matrixMulCUDA<<<blocksPerGrid, threadsPerBlock>>>(d_matA, d_matB, d_matC, matrixSize);
    cudaDeviceSynchronize();

    cudaMemcpy(matC_gpu, d_matC, bytes, cudaMemcpyDeviceToHost);

    end = high_resolution_clock::now();

    cout << "[Matrix Multiplication - GPU] Time: "
         << duration_cast<milliseconds>(end - start).count() << " ms\n";

    // ---------------- CLEANUP ----------------
    delete[] h_a;
    delete[] h_b;
    delete[] h_c_cpu;
    delete[] h_c_gpu;
    delete[] matA;
    delete[] matB;
    delete[] matC_cpu;
    delete[] matC_gpu;

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    cudaFree(d_matA);
    cudaFree(d_matB);
    cudaFree(d_matC);

    return 0;
}

Overwriting assignment.cu


In [6]:
!nvcc assignment.cu -o assignment

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [7]:
!./assignment

[Vector Addition - CPU] Time: 78 ms
[Vector Addition - GPU] Time: 72 ms

[Matrix Multiplication - CPU] Time: 8217 ms
[Matrix Multiplication - GPU] Time: 14 ms
